# RNG seeding control: draw a new task on reset

## Point

Episodes within a task share a map, so a route discovered in an earlier episode can help in a later one. Once the task ends, call `env.reset(options={"regenerate_map": True})` to start a new task with a newly drawn map. The new task does not continue the previous one.

> **Warning: reset and seeding behavior differs between environments.** Take extreme care to verify that these settings produce the configuration you intend. Some environments randomize only the starting state; others regenerate maps, goals, or dynamics on reset. They may use separate random generators or support different reset options. A fixed seed can repeat more than the map, and `seed=None` does not guarantee that the task stays fixed.
>
> Before running an experiment, inspect the base environment's reset behavior and check short rollouts across both episode and task boundaries. Verify which maps, goals, starting states, and random outcomes stay fixed or change, and confirm that replaying seeds reproduces what you expect. Choose the seed mode deliberately: `per_episode` varies episode seeds, `per_task` repeats a seed within each task, and `constant` repeats it across tasks too. Public `reset(seed=...)` restarts the generator in every mode; do not put a seed in `episode_reset_options`. The wrapper cannot guarantee the intended task configuration for every environment.

## Objective

Like the [commute example](00_episodes_to_tasks.ipynb), this environment uses public `reset(options=...)` to request a new task instance only at task start. Every episode receives an explicit seed; task generation is controlled separately by the reset options. This gives repeated attempts a stable task to learn about. A map discovered during an early episode can then help the agent improve its route in a later episode, which is the learning progress we want to measure. This notebook requires Python 3.12+ and the `examples` extra.

`Procedural-FrozenLake-v1` builds a map on reset and keeps it until a reset passes `options={"regenerate_map": True}`. Set `episode_reset_options={"regenerate_map": False}` and call `env.reset(options={"regenerate_map": True})` when starting each task. The public reset options update and override the episode defaults for that initial reset only. Internal resets use the original episode options, so those attempts share the map. Without the public override, a new task keeps the existing map.

`map_seed` makes map draws reproducible for comparison. Seed `action_space` after `reset` to reproduce sampled actions; that seed is separate from the map. The wrapper uses one RNG to generate episode seeds. An integer passed to public `reset(seed=...)` reseeds that RNG; `seed=None` leaves it running. By default, `episode_seed_mode="per_episode"` draws a seed at every episode start, including the first episode of each task. Episode lengths and environment random draws do not affect the seed stream, but extra episodes consume extra seeds and therefore affect the next task's seed. Procedural FrozenLake uses its own map generator rather than that seed to draw the map. Reproducible sampling does not carry task state or agent memory between tasks.

The cell uses a 5×5 lake and two episodes per task. The same `map_seed` and action seed match, including the board on the reset frame between episodes. The next task's board differs. A different `map_seed` changes the boards, and a different action seed changes the actions while leaving the boards in place.

The random actions here demonstrate map persistence and replay, not route learning. `emit_map=True` exposes the board in `info` for inspection; the action sampler does not read it. In an experiment where the agent should infer the map, keep that inspection data out of its inputs. The wrapper preserves `info`, so it does not hide a map that the base environment exposes.

Procedural FrozenLake 1.0.0 shares nested map metadata between returned `info` dictionaries. The small adapter below deep-copies each result so saved frames remain independent. It runs inside Gymnasium's normal environment checker, which stays enabled.


In [ ]:
from copy import deepcopy
from dataclasses import replace

import gymnasium as gym
import procedural_frozenlake  # registers Procedural-FrozenLake-v1

from repeat_task_gym import RepeatTaskEnv


class IndependentInfoFrozenLake(procedural_frozenlake.ProceduralFrozenLakeEnv):
    """Return independent info snapshots, including the nested map metadata."""

    def reset(self, *, seed=None, options=None):
        observation, info = super().reset(seed=seed, options=options)
        return observation, deepcopy(info)

    def step(self, action):
        observation, reward, terminated, truncated, info = super().step(action)
        return observation, reward, terminated, truncated, deepcopy(info)


# Copy the registration so Gymnasium checks the adapted environment itself.
# The original registration and its default wrappers remain intact.
lake_spec = replace(
    gym.spec("Procedural-FrozenLake-v1"), entry_point=IndependentInfoFrozenLake,
)


def rollout(map_seed, action_seed):
    env = RepeatTaskEnv(
        gym.make(
            lake_spec,
            map_seed=map_seed,
            emit_map=True,
            width=5,
            height=5,
            min_border=1,
            max_border=1,
            shoreline_jaggedness=0,
            max_episode_steps=4,
        ),
        max_task_episodes=2,
        episode_reset_options={"regenerate_map": False},
    )
    rows = []
    boards = []
    try:
        _, info = env.reset(seed=0, options={"regenerate_map": True})
        env.action_space.seed(action_seed)
        boards.append(tuple(info["map"]["board"]))
        for _ in range(20):
            action = int(env.action_space.sample())
            observation, reward, terminated, truncated, info = env.step(action)
            board = tuple(info["map"]["board"])
            rows.append(
                {
                    "observation": int(observation["observation"]),
                    "episode_terminated": observation["episode_terminated"],
                    "episode_truncated": observation["episode_truncated"],
                    "reward": float(reward),
                    "terminated": terminated,
                    "truncated": truncated,
                    "action": action,
                    "board": board,
                }
            )
            if terminated or truncated:
                _, info = env.reset(options={"regenerate_map": True})  # Request the next task/map.
                boards.append(tuple(info["map"]["board"]))
                break
    finally:
        env.close()
    return rows, boards


first, boards = rollout(0, 1)
replay, replay_boards = rollout(0, 1)
assert first == replay
assert boards == replay_boards
assert all(row["board"] == boards[0] for row in first)
continued = [
    index
    for index, row in enumerate(first)
    if (row["episode_terminated"] or row["episode_truncated"])
    and not row["terminated"] and not row["truncated"]
]
assert continued
reset_frame = first[continued[0] + 1]
assert not reset_frame["episode_terminated"] and not reset_frame["episode_truncated"]
assert first[-1]["truncated"] and not first[-1]["terminated"]
assert boards[0] != boards[1]

_, other_boards = rollout(1, 1)
assert other_boards != boards
moved, same_boards = rollout(0, 2)
assert same_boards == boards
assert [row["action"] for row in moved] != [row["action"] for row in first]

print("First task:")
print("\n".join(boards[0]))
print("Next task:")
print("\n".join(boards[1]))
print(f"Reproduced {len(first)} frames. A new map seed changes the boards.")


## Choose episode randomness independently of task options

All modes draw from one wrapper RNG. The default `episode_seed_mode="per_episode"` draws at every episode start. `"per_task"` draws only at task start. `"constant"` draws only on the first reset or after explicit reseeding. Whenever no draw is due, the underlying reset receives the last drawn seed again.

In every mode, public `reset(seed=...)` with an integer reseeds the wrapper RNG and immediately draws a new episode seed. The environment receives the drawn seed, not the supplied integer. Subsequent seedless resets continue the stream in per-episode and per-task modes, or retain the chosen seed in constant mode. Seeds belong in the public reset call; `episode_reset_options` accepts only environment options and rejects a `seed` entry.

For an environment that generates its layout from the episode seed on every reset, such as BipedalWalker, per-task mode can replay the layout within a task. Constant mode can replay it across tasks too. Both also repeat seeded initial conditions; neither selectively freezes only the layout. Verify the actual behavior of your environment.

The next cell compares two episodes in each of two CartPole tasks, then explicitly reseeds before a third task. It checks that per-episode mode varies starts, per-task mode repeats starts within each task, and constant mode repeats starts across tasks until reseeded. Replaying the public reset seeds reproduces the same results.


In [ ]:
import numpy as np


def episode_starts(mode, task_seeds=(0, None)):
    env = RepeatTaskEnv(
        gym.make("CartPole-v1", max_episode_steps=1),
        max_task_episodes=2,
        episode_seed_mode=mode,
    )
    starts = []
    try:
        for seed in task_seeds:
            first, _ = env.reset(seed=seed)
            env.step(0)  # End episode one.
            second, _, _, _, _ = env.step(0)  # Reset frame for episode two.
            assert env.step(0)[3]  # End episode two and the task.
            starts.append([first["observation"], second["observation"]])
        return np.array(starts)
    finally:
        env.close()


for mode in ("per_episode", "per_task", "constant"):
    starts = episode_starts(mode)
    np.testing.assert_array_equal(starts, episode_starts(mode))
    for task_starts in starts:
        assert np.array_equal(task_starts[0], task_starts[1]) == (mode != "per_episode")
    assert np.array_equal(starts[0, 0], starts[1, 0]) == (mode == "constant")

    reseeded = episode_starts(mode, task_seeds=(0, None, 1, None))
    np.testing.assert_array_equal(reseeded[:2], starts)
    np.testing.assert_array_equal(reseeded[2:], episode_starts(mode, task_seeds=(1, None)))
    assert not np.array_equal(reseeded[0, 0], reseeded[2, 0])
    print(
        f"{mode}: same starts within a task = {mode != 'per_episode'}; "
        f"same starts across tasks = {mode == 'constant'}; explicit reseeding changes them"
    )


## Preserve the task, then control its draws

Keeping the board fixed within a task makes knowledge from an earlier attempt useful on a later one. A public `env.reset(options={"regenerate_map": True})` regenerates it for an independent task; an agent must discard the completed task's route memory. Fresh draws can repeat; these particular seeded boards differ.

Matching map and action seeds checks reproducibility. Comparing agents instead holds the map draws fixed while allowing their actions to differ. The wrapper seed stream, map generator, and action sampler have distinct roles here: a wrapper seed cannot control randomness that the base environment manages separately.

Next: [04 — Gymnasium composition](04_gymnasium_composition.ipynb) records attempts and whole tasks so their returns can be interpreted together.
